# Análisis de Explicabilidad con SHAP

Usamos SHAP (SHapley Additive exPlanations) para entender qué variables impulsan las predicciones del modelo ganador.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from src.data.preprocess import dividir_datos
from src.models.evaluate import calcular_shap_values

modelo = joblib.load('models/best_model.pkl')
df = pd.read_parquet('data/processed/dataset_features.parquet')
X_train, X_test, y_train, y_test = dividir_datos(df)
muestra = X_test.sample(500, random_state=42)
print(f"Muestra para SHAP: {muestra.shape}")

## Summary Plot (importancia global)

In [ ]:
shap_values, explainer, X_trans = calcular_shap_values(modelo, muestra)
# Obtener nombres de features del preprocesador
feature_names = modelo.named_steps['prep'].get_feature_names_out().tolist()
shap.summary_plot(shap_values, X_trans, feature_names=feature_names, max_display=15)

## Waterfall — perfil individual (primer registro)

In [ ]:
import shap
clasificador = modelo.named_steps['modelo']
preprocesador = modelo.named_steps['prep']
X_trans_arr = preprocesador.transform(muestra)

# Para modelos de árbol, usar TreeExplainer directamente
try:
    exp = shap.TreeExplainer(clasificador)
    sv = exp(X_trans_arr)
    shap.waterfall_plot(sv[0])
except Exception as e:
    print(f"Waterfall no disponible para este modelo: {e}")

## Dependence Plot — variable más importante

In [ ]:
# Top feature por importancia SHAP media
importancias = np.abs(shap_values).mean(axis=0)
top_idx = np.argmax(importancias)
top_feature = feature_names[top_idx] if feature_names else f"feature_{top_idx}"
print(f"Variable más importante: {top_feature}")
shap.dependence_plot(top_idx, shap_values, X_trans, feature_names=feature_names)

## Beeswarm Plot — distribución de impactos

In [ ]:
try:
    exp = shap.TreeExplainer(clasificador)
    sv = exp(X_trans_arr)
    shap.plots.beeswarm(sv, max_display=12)
except Exception as e:
    print(f"Beeswarm no disponible: {e}")